In [1]:
# 自动重载外部文件的更新
import numpy as np
%load_ext autoreload
%autoreload 2

# 添加项目路径至path
import os
import sys
currentPath = os.path.join(os.getcwd(),"machinelearningIntro","机器学习实践/关联规则")
sys.path.append(currentPath)

In [2]:
import pandas as pd
import numpy as np
import math

In [3]:
datadir = r"D:\PythonEx\machinelearningIntro\机器学习实践\关联规则\data\ratings.csv"
df = pd.read_csv(datadir)

In [4]:
len(df)

1000209

In [5]:
df = df.head(100)

In [6]:
df

,userid,moveid,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
95,2,2490,3,978299966
96,2,1834,4,978298813
97,2,3471,5,978298814
98,2,589,4,978299773


In [10]:
# 计算物品项集，计算物品总数
distinct_item_list = list(df["moveid"].drop_duplicates().sort_values().reset_index(drop=True))
movie_count = len(distinct_item_list)

In [7]:
df1 = pd.get_dummies(df[['userid','moveid']],columns=["moveid"],prefix="",prefix_sep="",dtype=bool)
df1

,userid,1,48,110,150,260,292,368,434,527,...,3114,3147,3186,3255,3256,3257,3408,3468,3471,3578
0,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
4,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
96,2,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
97,2,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
98,2,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [8]:
%%time
inverted_table = df1.groupby("userid").agg(sum)

CPU times: total: 0 ns
Wall time: 0 ns


In [9]:
inverted_table

,1,48,110,150,260,292,368,434,527,531,...,3114,3147,3186,3255,3256,3257,3408,3468,3471,3578
userid,,,,,,,,,,,,,,,,,,,,,
1,1,1,0,1,1,0,0,0,1,1,...,1,0,1,0,0,0,1,0,0,0
2,0,0,1,0,0,1,1,1,0,0,...,0,1,0,1,1,1,0,1,1,1


In [11]:
inverted_table_arr = np.array(inverted_table)
inverted_table_arr

array([[1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0,
        1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0,
        0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0,
        1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0,
        0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1,
        1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1,
        0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1,
        1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1]], dtype=int64)

In [12]:
inverted_table_bin = []
for i in range(len(inverted_table_arr)):
    num = int("".join([str(j) for j in inverted_table_arr[i]]),2)
    inverted_table_bin.append(num)

In [15]:
# 计算同时喜欢同两样物品的人数的方法(物品共现矩阵用)
def like_the_same_two_items_users_count(inverted_table_bin,item1_index,item2_index):
    movie_number = movie_count
    item1_mask = 1 << (movie_number - item1_index)
    item2_mask = 1<<(movie_number - item2_index)
    and_mask = item1_mask|item2_mask

    counter = 0
    for i in inverted_table_bin:
        if i & and_mask == and_mask:
            counter = counter + 1

    return counter

In [16]:
like_the_same_two_items_users_count(inverted_table_bin, 1, 2)

1

In [17]:
%%time
co_occurrence1 = pd.DataFrame([[like_the_same_two_items_users_count(inverted_table_bin,i,j) for i in range(movie_count)] for j in range(movie_count)],columns=distinct_item_list,index=distinct_item_list)
co_occurrence1


CPU times: total: 15.6 ms
Wall time: 15.4 ms


,1,48,110,150,260,292,368,434,527,531,...,3114,3147,3186,3255,3256,3257,3408,3468,3471,3578
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
48,0,1,1,0,1,1,0,0,0,1,...,0,1,0,1,0,0,0,1,0,0
110,0,1,1,0,1,1,0,0,0,1,...,0,1,0,1,0,0,0,1,0,0
150,0,0,0,1,0,0,1,1,1,0,...,1,0,1,0,1,1,1,0,1,1
260,0,1,1,0,1,1,0,0,0,1,...,0,1,0,1,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3257,0,0,0,1,0,0,1,1,1,0,...,1,0,1,0,1,1,1,0,1,1
3408,0,0,0,1,0,0,1,1,1,0,...,1,0,1,0,1,1,1,0,1,1
3468,0,1,1,0,1,1,0,0,0,1,...,0,1,0,1,0,0,0,1,0,0
3471,0,0,0,1,0,0,1,1,1,0,...,1,0,1,0,1,1,1,0,1,1


In [18]:
%%time
# 计算每个物品共有几人购买
item_selected_list = inverted_table.agg(sum,axis=0)

CPU times: total: 0 ns
Wall time: 0 ns


In [27]:
item_selected_list

1       1
48      1
110     1
150     1
260     1
       ..
3257    1
3408    1
3468    1
3471    1
3578    1
Length: 99, dtype: int64

In [35]:
print(item_selected_list["110"])

1


In [36]:
def cal_similarity(i,j):
        return co_occurrence1[i][j]/math.sqrt(item_selected_list[str(i)]*item_selected_list[str(j)])

In [56]:
%%time
item_similarity_matrix = pd.DataFrame([[cal_similarity(i,j) for i in distinct_item_list] for j in distinct_item_list],columns=distinct_item_list,index=distinct_item_list)

CPU times: total: 141 ms
Wall time: 139 ms


In [39]:
item_similarity_matrix

,1,48,110,150,260,292,368,434,527,531,...,3114,3147,3186,3255,3256,3257,3408,3468,3471,3578
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
48,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
110,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
150,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,...,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0
260,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3257,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,...,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0
3408,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,...,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0
3468,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3471,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,...,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0


In [44]:
# 向任意用户x推荐物品
def recommend_to_userX(x):
    x_has_brought_item_list = inverted_table.loc[x,]
    x_has_not_brought_item_list = [i for i in distinct_item_list if i not in x_has_brought_item_list]
    item_score = 0
    item_has_not_brought_score_dict = dict()
    for item_not_brought in x_has_not_brought_item_list:
        for item_brought in x_has_brought_item_list:
            # 查询用户既往对该物品的打分，如果为空就设为0
            rating = df.loc[(df["userid"] == x) & (df['moveid'] == item_brought),'rating']
            if len(rating) == 1:
                item_brought_ranking = int(rating)
            else:
                item_brought_ranking = 0
            item_score += item_brought_ranking * (item_similarity_matrix.iloc[item_brought][item_not_brought])
        item_has_not_brought_score_dict[item_not_brought] = item_score

    return item_has_not_brought_score_dict

In [47]:
item_has_not_brought_score_dict = recommend_to_userX(1)

In [53]:
pd.DataFrame(item_has_not_brought_score_dict.values(),index=item_has_not_brought_score_dict.keys()).sort_values(by=0,ascending=False)[:5]

,0
3578,14045.0
3471,14045.0
3468,14045.0
3408,13780.0
3257,13780.0
